In [2]:
import sys, os, json, torch, random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import spearmanr
from nltk.translate.bleu_score import corpus_bleu
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

sys.path.append(os.path.abspath(os.path.join('..')))
from caption import caption_image_beam_search

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH   = '../checkpoints/BEST_checkpoint_coco_5_cap_per_img_5_min_word_freq.pth.tar'
WORD_MAP_PATH= '../dataset/WORDMAP_coco_5_cap_per_img_5_min_word_freq.json'
VAL_DIR      = '../dataset/val2014'
DATASET_JSON = '../dataset/caption_datasets/dataset_coco.json'
# Official COCO instance annotations — ground truth niezależny od opisów tekstowych
INSTANCES_JSON = '../dataset/annotations/instances_val2014.json'
BEAM_SIZE    = 3
SAMPLES_PER_CLASS = 30

with open(WORD_MAP_PATH, 'r') as f: word_map = json.load(f)
rev_word_map = {v: k for k, v in word_map.items()}
SPECIALS = {word_map['<start>'], word_map['<end>'], word_map['<pad>']}

checkpoint = torch.load(MODEL_PATH, map_location=str(DEVICE), weights_only=False)
encoder = checkpoint['encoder'].to(DEVICE).eval()
decoder = checkpoint['decoder'].to(DEVICE).eval()

# Classes span the full rarity spectrum; distractors are visually similar objects
# that the model might confuse with the target (semantic fallback candidates)
RAW_CLASSES = {
    'toaster'  : {'targets': {'toaster','toasters'},                          'distractors': {'microwave','oven','box'}},
    'frisbee'  : {'targets': {'frisbee','frisbees'},                          'distractors': {'plate','disc','ball'}},
    'mouse'    : {'targets': {'mouse'},                                       'distractors': {'keyboard','laptop','remote'}},
    'suitcase' : {'targets': {'suitcase','suitcases','luggage'},              'distractors': {'bag','backpack','box'}},
    'carrot'   : {'targets': {'carrot','carrots'},                            'distractors': {'banana','hotdog','vegetable'}},
    'backpack' : {'targets': {'backpack','backpacks','knapsack'},             'distractors': {'bag','suitcase','purse'}},
    'bench'    : {'targets': {'bench','benches'},                             'distractors': {'chair','table','seat'}},
    'couch'    : {'targets': {'couch','sofa'},                                'distractors': {'chair','bed','bench'}},
    'dog'      : {'targets': {'dog','dogs','puppy'},                          'distractors': {'cat','bear','animal'}},
    'car'      : {'targets': {'car','cars','automobile'},                     'distractors': {'truck','bus','van'}},
    'chair'    : {'targets': {'chair','chairs'},                              'distractors': {'bench','couch','stool'}},
    'person'   : {'targets': {'person','man','woman','people','boy','girl'},  'distractors': {'dog','cat','mannequin'}},
}

# Drop any class whose every target token was removed by min_word_freq filtering
TARGET_CLASSES = {}
print("=== Vocabulary check ===")
for obj, sem in RAW_CLASSES.items():
    vt = {w for w in sem['targets']    if w in word_map}
    vd = {w for w in sem['distractors'] if w in word_map}
    if vt:
        TARGET_CLASSES[obj] = {'targets': vt, 'distractors': vd}
        dropped_t = sem['targets'] - vt
        if dropped_t: print(f"  [{obj}] dropped targets (OOV): {dropped_t}")
    else:
        print(f"  [CRITICAL] '{obj}' — all targets OOV, class removed.")

print(f"\nFinal class count: {len(TARGET_CLASSES)}")

=== Vocabulary check ===
  [backpack] dropped targets (OOV): {'knapsack'}

Final class count: 12


In [3]:
# Load official COCO instance annotations for val2014
with open(INSTANCES_JSON, 'r') as f:
    instances = json.load(f)

# Build lookup: coco_category_name -> set of image_ids that contain that object
coco_cat_id   = {c['name']: c['id']   for c in instances['categories']}
coco_img_ids  = {}  # category_name -> set of image_ids
for ann in instances['annotations']:
    cat_name = next(c['name'] for c in instances['categories'] if c['id'] == ann['category_id'])
    coco_img_ids.setdefault(cat_name, set()).add(ann['image_id'])

coco_id_to_filename = {img['id']: img['file_name'] for img in instances['images']}

# Load Karpathy test split and its captions (for BLEU-4 references)
with open(DATASET_JSON, 'r') as f:
    karpathy = json.load(f)['images']

# Map filename -> list of 5 reference token lists (for BLEU-4)
karpathy_refs = {}
karpathy_test_filenames = set()
for img in karpathy:
    if img['split'] == 'test':
        karpathy_refs[img['filename']] = [s['tokens'] for s in img['sentences']]
        karpathy_test_filenames.add(img['filename'])

# For each class: collect test images confirmed by bounding-box annotations
# This eliminates selection tautology — ground truth is visual, not textual
diagnostic_pool = {obj: [] for obj in TARGET_CLASSES}

for obj in TARGET_CLASSES:
    # COCO category name may differ slightly; handle 'person' = 'person', etc.
    coco_name = obj  # all 12 chosen classes match COCO category names exactly
    if coco_name not in coco_img_ids:
        print(f"WARNING: '{coco_name}' not found in COCO categories.")
        continue
    for img_id in coco_img_ids[coco_name]:
        filename = coco_id_to_filename.get(img_id)
        if filename is None: continue
        if filename not in karpathy_test_filenames: continue  # only Karpathy test split
        filepath = os.path.join(VAL_DIR, filename)
        if not os.path.exists(filepath): continue
        diagnostic_pool[obj].append((filepath, karpathy_refs[filename]))

for obj, pool in diagnostic_pool.items():
    print(f"  {obj:>10}: {len(pool):4d} candidate images in test split")

FileNotFoundError: [Errno 2] No such file or directory: '../dataset/annotations/instances_val2014.json'

In [ ]:
# Stratified random sample per class
sampled = {}
for obj, pool in diagnostic_pool.items():
    if len(pool) < SAMPLES_PER_CLASS:
        print(f"WARNING: {obj} has only {len(pool)} images; using all.")
    sampled[obj] = random.sample(pool, min(SAMPLES_PER_CLASS, len(pool)))

# Training frequency: count train images that contain the target object
# Uses Karpathy train split + target token set (text-level, consistent with model's training signal)
train_imgs = [img for img in karpathy if img['split'] == 'train']
train_freq = {obj: 0 for obj in TARGET_CLASSES}

for img in tqdm(train_imgs, desc="Computing train frequencies", leave=False):
    tokens = set(w for s in img['sentences'] for w in s['tokens'])
    for obj, sem in TARGET_CLASSES.items():
        if tokens & sem['targets']:
            train_freq[obj] += 1

print("\n=== Training image frequencies ===")
for obj, freq in sorted(train_freq.items(), key=lambda x: x[1]):
    print(f"  {obj:>10}: {freq}")

In [ ]:
audit_rows = []
bleu_data  = {}  # obj -> (refs_list, hyps_list)

for obj, pool in tqdm(sampled.items(), desc="Classes"):
    targets    = TARGET_CLASSES[obj]['targets']
    distractors= TARGET_CLASSES[obj]['distractors']
    refs_acc, hyps_acc = [], []

    for filepath, refs in pool:
        seq, _ = caption_image_beam_search(encoder, decoder, filepath, word_map, BEAM_SIZE)
        hyp = [rev_word_map[w] for w in seq if w not in SPECIALS]
        hyp_set = set(hyp)

        # Three-way classification: priority order matters.
        # Success must be checked before distractor to avoid penalising
        # captions that correctly name the object AND mention a related one.
        if   hyp_set & targets:     category = 'Success'
        elif hyp_set & distractors: category = 'Hallucination'
        else:                       category = 'Omission'

        audit_rows.append({
            'object'   : obj,
            'frequency': train_freq[obj],
            'filepath' : filepath,
            'hypothesis': ' '.join(hyp),
            'category' : category,
        })
        refs_acc.append(refs)
        hyps_acc.append(hyp)

    bleu_data[obj] = corpus_bleu(refs_acc, hyps_acc, weights=(.25,.25,.25,.25))

df = pd.DataFrame(audit_rows)

# Per-class summary table
stats = (df.groupby('object')
           .agg(train_freq=('frequency','first'),
                n=('filepath','count'),
                successes=('category', lambda x: (x=='Success').sum()))
           .reset_index())
stats['recall']  = stats['successes'] / stats['n'] * 100
stats['bleu4']   = stats['object'].map(bleu_data)
stats = stats.sort_values('train_freq').reset_index(drop=True)

display(stats[['object','train_freq','n','successes','recall','bleu4']])

In [ ]:
# Two Spearman tests on the same N points -> Bonferroni correction: alpha/2 = 0.025
ALPHA = 0.05
ALPHA_CORRECTED = ALPHA / 2

r_rec, p_rec   = spearmanr(stats['train_freq'], stats['recall'])
r_b4,  p_b4    = spearmanr(stats['train_freq'], stats['bleu4'])

print(f"N = {len(stats)} classes | Bonferroni-corrected significance threshold: p < {ALPHA_CORRECTED}\n")
print(f"Train frequency vs Object Recall : r = {r_rec:+.3f},  p = {p_rec:.4f}  "
      f"{'[SIGNIFICANT]' if p_rec < ALPHA_CORRECTED else '[not significant]'}")
print(f"Train frequency vs BLEU-4        : r = {r_b4:+.3f},  p = {p_b4:.4f}  "
      f"{'[SIGNIFICANT]' if p_b4  < ALPHA_CORRECTED else '[not significant]'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- LEFT: Proportional error distribution ---
pivot = (df.groupby(['object','category'])
           .size()
           .unstack(fill_value=0)
           .reindex(columns=['Success','Hallucination','Omission'], fill_value=0))
pivot['_freq'] = pivot.index.map(train_freq)
pivot = pivot.sort_values('_freq').drop(columns='_freq')
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

pivot_pct.plot(kind='bar', stacked=True, ax=axes[0],
               color=['#2ecc71','#e74c3c','#95a5a6'],
               edgecolor='black', width=0.75)

axes[0].set_title('Error Distribution by Object Class\n(sorted: rare → frequent)', fontsize=13)
axes[0].set_ylabel('Percentage of test images (%)')
axes[0].set_xlabel('Object class')
axes[0].set_ylim(0, 100)
axes[0].legend(title='Outcome', bbox_to_anchor=(1.01, 1), loc='upper left')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Only label segments large enough to be readable
for container in axes[0].containers:
    labels = [f'{v:.0f}%' if v >= 8 else '' for v in container.datavalues]
    axes[0].bar_label(container, labels=labels, label_type='center',
                      fontweight='bold', color='white', fontsize=9)

# --- RIGHT: Scatter — training frequency vs recall rate ---
axes[1].scatter(stats['train_freq'], stats['recall'],
                color='#2980b9', s=80, zorder=3)

for _, row in stats.iterrows():
    axes[1].annotate(row['object'],
                     (row['train_freq'], row['recall']),
                     textcoords='offset points', xytext=(5, 3), fontsize=9)

# Trend line
m, b = np.polyfit(stats['train_freq'], stats['recall'], 1)
x_line = np.linspace(stats['train_freq'].min(), stats['train_freq'].max(), 100)
axes[1].plot(x_line, m * x_line + b, 'r--', alpha=0.6, linewidth=1.5)

axes[1].set_title(f'Training Frequency vs Object Recall\n'
                  f'Spearman r = {r_rec:+.3f}, p = {p_rec:.4f}', fontsize=13)
axes[1].set_xlabel('Number of training images containing object')
axes[1].set_ylabel('Object Recall Rate (%)')
axes[1].grid(linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('exp4_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def plot_attention(image_path, seq, alphas, rev_word_map, title, max_words=8):
    img = Image.open(image_path).convert('RGB').resize((256, 256), Image.LANCZOS)
    words = [rev_word_map[i] for i in seq]
    clean_idx = [i for i, w in enumerate(words) if w not in {'<start>','<end>','<pad>'}]

    grid = int(np.sqrt(alphas.shape[-1]))
    alpha_np = alphas.view(-1, grid, grid).cpu().detach().numpy()

    n = min(len(clean_idx), max_words)
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 3))
    fig.suptitle(title, fontsize=12, fontweight='bold')

    for plot_i, seq_i in enumerate(clean_idx[:n]):
        ax = axes[plot_i]
        ax.imshow(img)
        heat = Image.fromarray(alpha_np[seq_i]).resize((256, 256), Image.BILINEAR)
        ax.imshow(heat, alpha=0.55, cmap='jet')
        ax.set_title(words[seq_i], fontsize=10,
                     backgroundcolor='#f1c40f', fontweight='bold')
        ax.axis('off')

    plt.tight_layout()
    plt.show()


# Find the rarest class that has both a Success and a Hallucination case
candidate_classes = (
    stats.merge(
        df.groupby('object')['category']
          .apply(lambda x: set(x)).rename('cats'),
        on='object'
    )
    .query("cats.apply(lambda s: 'Success' in s and 'Hallucination' in s)")
    .sort_values('train_freq')
)

if candidate_classes.empty:
    print("No class has both Success and Hallucination cases. Try increasing SAMPLES_PER_CLASS.")
else:
    chosen = candidate_classes.iloc[0]['object']
    class_df = df[df['object'] == chosen]

    success_row = class_df[class_df['category'] == 'Success'].iloc[0]
    failure_row = class_df[class_df['category'] == 'Hallucination'].iloc[0]

    print(f"Comparative attention analysis — class: '{chosen}' "
          f"(train freq: {train_freq[chosen]})\n")

    for row, label in [(success_row, '[CONTROL — Success]'),
                       (failure_row, '[HALLUCINATION — Semantic Fallback]')]:
        print(f"{label}: {row['hypothesis']}")
        seq, alphas = caption_image_beam_search(
            encoder, decoder, row['filepath'], word_map, BEAM_SIZE)
        plot_attention(row['filepath'], seq, alphas, rev_word_map,
                       f"{label}\n{row['hypothesis']}")